In [ ]:
path_to_dataset = ""    #FIXME: Add path to dataset here
MAX_WORKERS = 3         #FIXME: Adjust based on your CPU cores / GPU VRAM capacity

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

torch.set_num_threads(1)
cv2.setNumThreads(1)
os.environ["OMP_NUM_THREADS"] = "1"

import sys
sys.path.append('..')

from Pipeline import Pipeline

OUTPUT_CSV_PATH = 'Results/notch_code_results.csv'

In [ ]:
pipeline = Pipeline()

# Gather all image paths recursively from subdirectories
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')
image_paths = []
for root, dirs, files in os.walk(path_to_dataset):
    for file in files:
        if file.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images to process.")

In [ ]:
def process_single_image(img_path):
    """Helper function to process one image and format rows for the dataset."""
    try:
        # Run the integrated pipeline function
        data = pipeline.process_and_code(img_path)
        if data is None:
            return []
            
        rows = []
        
        used_notches = data.get("used_notches", [])
        unused_notches = data.get("unused_notches", [])
        
        # 1. Flag the entire image for review if ANY of the *used* notches fell below the confidence threshold
        image_needs_review = any(n.get("needs_human_review", False) for n in used_notches)     

        # 2. Combine them into a single list with a tuple tracking their usage status
        all_valid_notches = [(n, True) for n in used_notches] + [(n, False) for n in unused_notches]

        # If no notches were found or border is missing, log the image entry
        if not all_valid_notches:
            rows.append({
                "image_filename": data["image_filename"],
                "img_width": data["img_width"],
                "img_height": data["img_height"],
                "border_present": data["border_present"],
                "notch_code": data["notch_code"],
                "image_needs_review": False,      
                "is_used": False,
                "shape": "none",
                "shape_confidence": 0.0,
                "slope_direction": "none",        
                "direction_confidence": 0.0,
                "needs_human_review": False,
                "yolo_prob": 0.0,
                "svm_notch_prob": 0.0,
                "xmin": 0, "ymin": 0, "xmax": 0, "ymax": 0
            })
        else:
            # Log each accepted notch
            for notch, is_used in all_valid_notches:
                xmin, ymin, xmax, ymax = notch['coords']
                rows.append({
                    "image_filename": data["image_filename"],
                    "img_width": data["img_width"],
                    "img_height": data["img_height"],
                    "border_present": data["border_present"],
                    "notch_code": data["notch_code"],
                    "image_needs_review": image_needs_review, # Applies to all rows for this specific image
                    "is_used": is_used,             
                    "shape": notch.get('shape', 'unknown'),
                    "shape_confidence": notch.get('shape_confidence', 0.0),
                    "slope_direction": notch.get('slope_direction', 'none'),      
                    "direction_confidence": notch.get('direction_confidence', 0.0),
                    "needs_human_review": notch.get('needs_human_review', False), # Specific to this notch
                    "yolo_prob": notch.get('yolo_prob', 0.0),
                    "svm_notch_prob": notch.get('svm_notch_prob', 0.0),
                    "xmin": xmin,
                    "ymin": ymin,
                    "xmax": xmax,
                    "ymax": ymax
                })
        return rows
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return []

In [ ]:
print("Warming up models to prevent thread collisions...")
# Create a blank black image
dummy_img = np.zeros((1024, 1024, 3), dtype=np.uint8)
_ = pipeline.process_image(dummy_img)
print("Warm-up complete!")
# ------------------------------

original_count = len(image_paths)
skipped_count = 0

if os.path.exists(OUTPUT_CSV_PATH):
    try:
        processed_df = pd.read_csv(OUTPUT_CSV_PATH, usecols=["image_filename"])
        processed_filenames = set(processed_df["image_filename"].dropna().tolist())
        
        # Filter the list
        image_paths_filtered = [p for p in image_paths if os.path.basename(p) not in processed_filenames]
        
        # Calculate how many were skipped
        skipped_count = original_count - len(image_paths_filtered)
        print(f"Found existing CSV! Skipped {skipped_count} already processed images.")
        
    except pd.errors.EmptyDataError:
        print("CSV is empty. Starting fresh.")
else:
    image_paths_filtered = image_paths
    pd.DataFrame(columns=[
        "image_filename", "img_width", "img_height", "border_present", 
        "notch_code", "image_needs_review", "is_used", "shape", 
        "shape_confidence", "slope_direction", "direction_confidence", 
        "needs_human_review", "yolo_prob", "svm_notch_prob", 
        "xmin", "ymin", "xmax", "ymax"
    ]).to_csv(OUTPUT_CSV_PATH, index=False)

executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
futures = {executor.submit(process_single_image, path): path for path in image_paths_filtered}

# Track how many images we've processed for auto-saving
processed_count = 0
buffer_rows = []
SAVE_EVERY_N_IMAGES = 50

try:
    for future in tqdm(as_completed(futures), total=original_count, initial=skipped_count, desc="Processing Images"):
        result_rows = future.result()
        if result_rows:
           buffer_rows.extend(result_rows)
        
        # Auto-save periodically just in case the notebook crashes entirely
        processed_count += 1
        if processed_count % SAVE_EVERY_N_IMAGES == 0:
            df_batch = pd.DataFrame(buffer_rows)
            df_batch.to_csv(OUTPUT_CSV_PATH, mode='a', header=False, index=False)
            buffer_rows = []
            
except KeyboardInterrupt:
    print("\nInterrupt received! Forcing shutdown...")
    # cancel_futures=True stops it from starting new images
    # wait=False tells it not to block the main thread waiting for current ones to finish
    executor.shutdown(wait=False, cancel_futures=True)
    print("Saving accumulated results...")

finally:
    # This ensures the final CSV is saved whether it finished normally or was interrupted
    if buffer_rows:
        df_batch = pd.DataFrame(buffer_rows)
        df_batch.to_csv(OUTPUT_CSV_PATH, mode='a', header=False, index=False)
    if os.path.exists(OUTPUT_CSV_PATH):
        total_rows = sum(1 for _ in open(OUTPUT_CSV_PATH)) - 1 # -1 for the header
        print(f"\nDone! Dataset now has {total_rows} rows in {OUTPUT_CSV_PATH}.")

**Analysis of result CSV**

In [ ]:
df = pd.read_csv(OUTPUT_CSV_PATH)

**Get all the diffrent possible notch codes** </br>
*Sloped notches lose direction as this can not be trusted (number of sloped_left = 0 after pipeline)*

In [ ]:
# 1. Get unique images
df_images = df.drop_duplicates(subset='image_filename').copy()

df_images['notch_code'] = df_images['notch_code'].str.replace('sloped_left', 'sloped', regex=False)
df_images['notch_code'] = df_images['notch_code'].str.replace('sloped_right', 'sloped', regex=False)

# 2. Group by code and calculate total instances and how many need review
code_analysis = df_images.groupby('notch_code').agg(
    total_instances=('image_filename', 'count'),
    needs_review_count=('image_needs_review', 'sum')
).reset_index()

# 3. Add a percentage column for easier reading
code_analysis['review_percentage'] = (code_analysis['needs_review_count'] / code_analysis['total_instances']) * 100

# Save to a new CSV file
sorted_analysis = code_analysis.sort_values(by='total_instances', ascending=False)
sorted_analysis.to_csv('Results/notch_code_frequencies.csv', index=False)

Get all the diffrent possible notch codes with the knowledge that circles appear in the corners and should thus be last in the code

In [ ]:
def fix_circle_direction(code):
    if not isinstance(code, str):
        return code
        
    # If it starts with circle, split by '-', reverse the list [::-1], and re-join
    if code.startswith('circle'):
        return '-'.join(code.split('-')[::-1])
    return code

In [ ]:
# 1. Get unique images
df_images = df.drop_duplicates(subset='image_filename').copy()

df_images['notch_code'] = df_images['notch_code'].str.replace('sloped_left', 'sloped', regex=False)
df_images['notch_code'] = df_images['notch_code'].str.replace('sloped_right', 'sloped', regex=False)

df_images['notch_code'] = df_images['notch_code'].apply(fix_circle_direction)

# 2. Group by code and calculate total instances and how many need review
code_analysis = df_images.groupby('notch_code').agg(
    total_instances=('image_filename', 'count'),
    needs_review_count=('image_needs_review', 'sum')
).reset_index()

# 3. Add a percentage column for easier reading
code_analysis['review_percentage'] = (code_analysis['needs_review_count'] / code_analysis['total_instances']) * 100

# Save to a new CSV file
sorted_analysis = code_analysis.sort_values(by='total_instances', ascending=False)
sorted_analysis.to_csv('Results/notch_code_frequencies_circle_fixed.csv', index=False)

**Shape metrics**

In [ ]:
# Filter out instances where no notch code was generated
filtered_df = df[df['notch_code'] != 'none']

# Group by shape and look at the average confidence
shape_metrics = filtered_df.groupby('shape').agg(
    count=('shape', 'count'),
    avg_shape_confidence=('shape_confidence', 'mean'),
    review_flagged=('needs_human_review', 'sum')
).reset_index()

print(shape_metrics)

In [ ]:
# Filter out instances where no notch code was generated
filtered_df = df[df['notch_code'] != 'none']

# Compare average probabilities between used and unused notches
grouping_analysis = filtered_df.groupby('is_used').agg(
    total_notches=('is_used', 'count'),
    avg_yolo_prob=('yolo_prob', 'mean'),
    avg_svm_prob=('svm_notch_prob', 'mean')
).reset_index()

print(grouping_analysis)

**Spatial heatmap of used and unused notches**

In [ ]:
import matplotlib.pyplot as plt

# Calculate the center (X, Y) of every notch
df['center_x'] = (df['xmin'] + df['xmax']) / 2
df['center_y'] = (df['ymin'] + df['ymax']) / 2

# Normalize coordinates (0.0 to 1.0) so different image sizes don't skew the data
df['norm_x'] = df['center_x'] / df['img_width']
df['norm_y'] = df['center_y'] / df['img_height']

# Separate used vs unused
used = df[df['is_used'] == True]
unused = df[df['is_used'] == False]

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(used['norm_x'], used['norm_y'], color='blue', alpha=0.5, label='Used Notches')
plt.scatter(unused['norm_x'], unused['norm_y'], color='red', alpha=0.5, label='Discarded (Noise)')
plt.title('Normalized Spatial Distribution of Detections')
plt.legend()
plt.gca().invert_yaxis() # Invert Y to match image coordinate systems
plt.show()